In [2]:
from datetime import datetime, timedelta
import time
import mysql.connector
import pandas as pd
from pytrends.request import TrendReq
import numpy as np
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
from pytrends.request import TrendReq
from datetime import date, datetime
from google.cloud import bigquery
import os

### Google trends data

In [4]:
# ------------------------------------------------------------------
# 1. Configuration & Constants
# ------------------------------------------------------------------
COUNTRY_MAP = {
    "US": "United_States",
    "IN": "India",
    "FR": "France",
    "NG": "Nigeria",
    "GB": "United_Kingdom",
    "CA": "Canada",
    "AU": "Australia",
    "BR": "Brazil",
    "MX": "Mexico",
    "ZA": "South_Africa",
    "KE": "Kenya",
    "SG": "Singapore",
    "AE": "United_Arab_Emirates",
    "CN": "China",
}

EXPANDED_CATEGORY_BATCHES = {
    "Nutrition_Diets": [
        ["diet", "nutrition", "vegan", "vegetarian", "plant based"],
        ["keto", "paleo", "low carb", "intermittent fasting"],
        ["detox", "superfood", "organic", "weight loss"],
        ["supplement", "protein powder", "whey", "creatine"],
        ["vitamin", "minerals", "probiotics", "functional food"],
    ],
    "Fitness_Wearables": [
        ["fitness", "exercise", "workout", "training", "gym"],
        ["yoga", "pilates", "aerobics", "hiit"],
        ["crossfit", "cardio", "running", "cycling"],
        ["wearable", "smartwatch", "fitness tracker", "garmin"],
        ["fitbit", "apple watch", "heart rate monitor", "step counter"],
    ],
    "Fashion_Beauty": [
        ["fashion", "clothing", "apparel", "style", "designer"],
        ["luxury fashion", "fast fashion", "streetwear", "athleisure"],
        ["skincare", "makeup", "cosmetics", "moisturizer"],
        ["anti aging", "haircare", "shampoo", "fragrance"],
        ["perfume", "beauty treatment", "sneakers", "jewelry"],
    ],
}



In [5]:
# ------------------------------------------------------------------
# 2. Connect to MySQL Database & Get Last Date
# ------------------------------------------------------------------
db_config = {
    "host": "localhost",
    "user": "root",
    "password": "!@Mstrong@1234",
    "database": "GDELTTrends",
}

connection = mysql.connector.connect(**db_config)

# Get the latest week start date from final_master_dataset
query = "SELECT MAX(STR_TO_DATE(Week_Start, '%Y-%m-%d')) AS last_date FROM final_master_dataset;"
last_date_df = pd.read_sql(query, connection)
connection.close()

last_recorded_date = last_date_df["last_date"].iloc[0]

if pd.isna(last_recorded_date):
    last_recorded_date = datetime(2026, 8, 28).date()
elif isinstance(last_recorded_date, (pd.Timestamp, datetime)):
    last_recorded_date = last_recorded_date.date()



C:\Users\Srivatsava\AppData\Local\Temp\ipykernel_7912\2178260422.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  last_date_df = pd.read_sql(query, connection)


In [6]:
# ------------------------------------------------------------------
# 3. Compute Google Trends Extraction Date Window
# ------------------------------------------------------------------

# Keep the database's latest recorded week as the reference point
google_trends_last_recorded_date = last_recorded_date

# Today
today = datetime.now().date()

# Sunday = 6 in Python's weekday() convention
# Find the current Sunday-starting week
current_week_start = today - timedelta(
    days=(today.weekday() + 1) % 7
)

# Current week is incomplete.
# Therefore, latest completed week is the previous Sunday.
google_trends_end_date = current_week_start - timedelta(days=7)

# ------------------------------------------------------------------
# Google Trends needs historical overlap for normalization
# ------------------------------------------------------------------

NORMALIZATION_OVERLAP_WEEKS = 8

google_trends_start_date = (
    google_trends_last_recorded_date
    - timedelta(days=NORMALIZATION_OVERLAP_WEEKS * 7)
)

# ------------------------------------------------------------------
# Check extraction range
# ------------------------------------------------------------------

if google_trends_last_recorded_date >= google_trends_end_date:

    print(
        f"✅ Google Trends data is already up to date as of "
        f"{google_trends_last_recorded_date}. "
        f"No completed weeks available to fetch."
    )

    exit()

# ------------------------------------------------------------------
# Google Trends timeframe
# ------------------------------------------------------------------

timeframe_str = (
    f"{google_trends_start_date.strftime('%Y-%m-%d')} "
    f"{google_trends_end_date.strftime('%Y-%m-%d')}"
)

print(f"📅 Last recorded week       : {google_trends_last_recorded_date}")
print(f"📅 Current week start       : {current_week_start}")
print(f"📅 Last completed week      : {google_trends_end_date}")
print(f"🔗 Normalization overlap    : {NORMALIZATION_OVERLAP_WEEKS} weeks")
print(f"🚀 Google Trends fetch range: {timeframe_str}\n")

📅 Last recorded week       : 2026-08-23
📅 Current week start       : 2026-09-20
📅 Last completed week      : 2026-09-13
🔗 Normalization overlap    : 8 weeks
🚀 Google Trends fetch range: 2026-06-28 2026-09-13



In [7]:
# ------------------------------------------------------------
# Create PyTrends session
# ------------------------------------------------------------
def create_pytrends_session():
    session = requests.Session()

    retries = Retry(
        total=3,
        connect=3,
        read=3,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"],
    )

    adapter = HTTPAdapter(max_retries=retries)

    session.mount("https://", adapter)
    session.mount("http://", adapter)

    return TrendReq(hl="en-US", tz=360, requests_args={"verify": True})


# ------------------------------------------------------------
# Fetch Google Trends with Sunday Weekly Resampling
# ------------------------------------------------------------

pytrends = create_pytrends_session()
all_new_records = []

for geo_code, country_name in COUNTRY_MAP.items():
    print(f"\n========== {country_name} ({geo_code}) ==========")

    for cat_name, batches in EXPANDED_CATEGORY_BATCHES.items():
        print(f"\nFetching category: {cat_name}")
        category_batch_dfs = []

        for batch_idx, keyword_batch in enumerate(batches):
            success = False

            for attempt in range(1, 4):
                try:
                    print(
                        f"   Batch {batch_idx + 1}/{len(batches)} "
                        f"| Attempt {attempt}"
                    )

                    pytrends.build_payload(
                        keyword_batch, geo=geo_code, timeframe=timeframe_str
                    )

                    df_raw = pytrends.interest_over_time()

                    if df_raw.empty:
                        print(
                            f"   ⚠️ Empty response for batch {batch_idx + 1}"
                        )
                        break

                    if "isPartial" in df_raw.columns:
                        df_raw = df_raw.drop(columns=["isPartial"])

                    batch_cols = [
                        kw for kw in keyword_batch if kw in df_raw.columns
                    ]

                    if not batch_cols:
                        print("   ⚠️ No keyword columns returned")
                        break

                    # --------------------------------------------------------
                    #  Resample daily data to weekly starting on Sunday
                    # --------------------------------------------------------
                    df_weekly = (
                        df_raw[batch_cols].resample("W-SUN").mean().round(2)
                    )

                    score_col = f"batch_{batch_idx}_score"
                    df_weekly[score_col] = df_weekly[batch_cols].mean(axis=1)

                    category_batch_dfs.append(df_weekly[[score_col]])

                    success = True
                    print("   ✓ Success")
                    break

                except Exception as e:
                    print(f"   ⚠️ Attempt {attempt} failed: {e}")
                    if attempt < 3:
                        wait_time = 5 * attempt
                        print(f"   Waiting {wait_time}s...")
                        time.sleep(wait_time)

            # Small delay between requests to avoid rate limits
            time.sleep(2)

        # ----------------------------------------------------
        # Combine batches for category
        # ----------------------------------------------------
        if category_batch_dfs:
            df_cat_merged = pd.concat(category_batch_dfs, axis=1)

            df_cat_merged["Search_Interest"] = df_cat_merged.mean(axis=1)

            max_val = df_cat_merged["Search_Interest"].max()

            if max_val > 0:
                df_cat_merged["Search_Interest"] = (
                    df_cat_merged["Search_Interest"] / max_val * 100
                ).round(2)

            df_cat_merged = df_cat_merged.reset_index().rename(
                columns={"date": "Week_Start"}
            )

            # Ensure date column is formatted as standard string (YYYY-MM-DD)
            df_cat_merged["Week_Start"] = pd.to_datetime(
                df_cat_merged["Week_Start"]
            ).dt.strftime("%Y-%m-%d")

            df_cat_merged["Country_Code"] = geo_code
            df_cat_merged["Country_Name"] = country_name
            df_cat_merged["Category"] = cat_name

            all_new_records.append(
                df_cat_merged[
                    [
                        "Week_Start",
                        "Country_Code",
                        "Country_Name",
                        "Category",
                        "Search_Interest",
                    ]
                ]
            )

            print(f"   ✓ Category completed: {cat_name}")
        else:
            print(f"   ❌ No data collected: {cat_name}")


========== United_States (US) ==========

Fetching category: Nutrition_Diets
   Batch 1/5 | Attempt 1
   ✓ Success
   Batch 2/5 | Attempt 1
   ✓ Success
   Batch 3/5 | Attempt 1
   ✓ Success
   Batch 4/5 | Attempt 1
   ✓ Success
   Batch 5/5 | Attempt 1
   ✓ Success
   ✓ Category completed: Nutrition_Diets

Fetching category: Fitness_Wearables
   Batch 1/5 | Attempt 1


KeyboardInterrupt: 

In [ ]:
# ------------------------------------------------------------------
# 5. Output Preview
# ------------------------------------------------------------------
if all_new_records:
    df_raw_incremental = pd.concat(all_new_records, ignore_index=True)
    df_raw_incremental["Week_Start"] = pd.to_datetime(df_raw_incremental["Week_Start"]).dt.strftime("%Y-%m-%d")
    print("\n" + "="*65)
    print(f"✅ INCREMENTAL DATA FETCHED: {len(df_raw_incremental):,} records")
    print(df_raw_incremental.head(10))
    print("="*65)


✅ INCREMENTAL DATA FETCHED: 504 records
   Week_Start Country_Code   Country_Name         Category  Search_Interest
0  2026-06-28           US  United_States  Nutrition_Diets           100.00
1  2026-07-05           US  United_States  Nutrition_Diets            93.09
2  2026-07-12           US  United_States  Nutrition_Diets            92.35
3  2026-07-19           US  United_States  Nutrition_Diets            84.94
4  2026-07-26           US  United_States  Nutrition_Diets            76.49
5  2026-08-02           US  United_States  Nutrition_Diets            73.18
6  2026-08-09           US  United_States  Nutrition_Diets            72.54
7  2026-08-16           US  United_States  Nutrition_Diets            71.69
8  2026-08-23           US  United_States  Nutrition_Diets            69.78
9  2026-08-30           US  United_States  Nutrition_Diets            71.53


In [ ]:
# saving data to file
df_raw_incremental.to_csv("New_fetched_data.csv",index=False)

NameError: name 'df_raw_incremental' is not defined

In [15]:
df_raw_incremental = pd.read_csv('New_fetched_data.csv')

### Normalization

In [16]:
# ------------------------------------------------------------
# 0. Global Configuration & Mappings
# ------------------------------------------------------------
COUNTRY_MAP = {
    "AE": "United_Arab_Emirates",
    "AU": "Australia",
    "BR": "Brazil",
    "CA": "Canada",
    "CN": "China",
    "FR": "France",
    "IN": "India",
    "KE": "Kenya",
    "MX": "Mexico",
    "NG": "Nigeria",
    "SG": "Singapore",
    "ZA": "South_Africa",
    "GB": "United_Kingdom",
    "US": "United_States"
}

EXPANDED_CATEGORY_BATCHES = {
    # Define your category batches dictionary here
}

timeframe_str = "today 3-m"


# ------------------------------------------------------------
# 1. Create PyTrends Session
# ------------------------------------------------------------
def create_pytrends_session():
    session = requests.Session()

    retries = Retry(
        total=3,
        connect=3,
        read=3,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"],
    )

    adapter = HTTPAdapter(max_retries=retries)

    session.mount("https://", adapter)
    session.mount("http://", adapter)

    return TrendReq(hl="en-US", tz=360, requests_args={"verify": True})


# ------------------------------------------------------------
# 2. Fetch Google Trends with Sunday Weekly Resampling
# ------------------------------------------------------------
pytrends = create_pytrends_session()
all_new_records = []

for geo_code, country_name in COUNTRY_MAP.items():
    print(f"\n========== {country_name} ({geo_code}) ==========")

    for cat_name, batches in EXPANDED_CATEGORY_BATCHES.items():
        print(f"\nFetching category: {cat_name}")
        category_batch_dfs = []

        for batch_idx, keyword_batch in enumerate(batches):
            success = False

            for attempt in range(1, 4):
                try:
                    print(
                        f"   Batch {batch_idx + 1}/{len(batches)} "
                        f"| Attempt {attempt}"
                    )

                    pytrends.build_payload(
                        keyword_batch, geo=geo_code, timeframe=timeframe_str
                    )

                    df_raw = pytrends.interest_over_time()

                    if df_raw.empty:
                        print(
                            f"   ⚠️ Empty response for batch {batch_idx + 1}"
                        )
                        break

                    if "isPartial" in df_raw.columns:
                        df_raw = df_raw.drop(columns=["isPartial"])

                    batch_cols = [
                        kw for kw in keyword_batch if kw in df_raw.columns
                    ]

                    if not batch_cols:
                        print("   ⚠️ No keyword columns returned")
                        break

                    # Resample daily data to weekly starting on Sunday
                    df_weekly = (
                        df_raw[batch_cols].resample("W-SUN").mean().round(2)
                    )

                    score_col = f"batch_{batch_idx}_score"
                    df_weekly[score_col] = df_weekly[batch_cols].mean(axis=1)

                    category_batch_dfs.append(df_weekly[[score_col]])

                    success = True
                    print("   ✓ Success")
                    break

                except Exception as e:
                    print(f"   ⚠️ Attempt {attempt} failed: {e}")
                    if attempt < 3:
                        wait_time = 5 * attempt
                        print(f"   Waiting {wait_time}s...")
                        time.sleep(wait_time)

            time.sleep(2)

        # Combine batches for category
        if category_batch_dfs:
            df_cat_merged = pd.concat(category_batch_dfs, axis=1)
            df_cat_merged["Search_Interest"] = df_cat_merged.mean(axis=1)

            max_val = df_cat_merged["Search_Interest"].max()
            if max_val > 0:
                df_cat_merged["Search_Interest"] = (
                    df_cat_merged["Search_Interest"] / max_val * 100
                ).round(2)

            df_cat_merged = df_cat_merged.reset_index().rename(
                columns={"date": "Week_Start"}
            )

            df_cat_merged["Week_Start"] = pd.to_datetime(
                df_cat_merged["Week_Start"]
            ).dt.strftime("%Y-%m-%d")

            df_cat_merged["Country_Code"] = geo_code
            df_cat_merged["Country_Name"] = country_name
            df_cat_merged["Category"] = cat_name

            all_new_records.append(
                df_cat_merged[
                    [
                        "Week_Start",
                        "Country_Code",
                        "Country_Name",
                        "Category",
                        "Search_Interest",
                    ]
                ]
            )

            print(f"   ✓ Category completed: {cat_name}")
        else:
            print(f"   ❌ No data collected: {cat_name}")


# ------------------------------------------------------------------
# 5. Output Preview & Save Raw Incremental Data
# ------------------------------------------------------------------
if all_new_records:
    df_raw_incremental = pd.concat(all_new_records, ignore_index=True)
    df_raw_incremental["Week_Start"] = pd.to_datetime(df_raw_incremental["Week_Start"]).dt.strftime("%Y-%m-%d")
    df_raw_incremental.to_csv("New_fetched_data.csv", index=False)
    
    print("\n" + "="*65)
    print(f"✅ INCREMENTAL DATA FETCHED: {len(df_raw_incremental):,} records")
    print(df_raw_incremental.head(10))
    print("="*65)


# ------------------------------------------------------------------
# 6. Load New Google Trends Data
# ------------------------------------------------------------------
df_incremental = pd.read_csv("New_fetched_data.csv")

# Basic cleaning
df_incremental["Week_Start"] = pd.to_datetime(df_incremental["Week_Start"], errors="coerce")
df_incremental["Search_Interest"] = pd.to_numeric(df_incremental["Search_Interest"], errors="coerce")

df_incremental = df_incremental.dropna(
    subset=["Week_Start", "Country_Code", "Category", "Search_Interest"]
)
df_incremental["Search_Interest"] = df_incremental["Search_Interest"].clip(lower=0)

# Fill Country_Name if missing in incremental load
if "Country_Name" not in df_incremental.columns or df_incremental["Country_Name"].isna().any():
    df_incremental["Country_Name"] = df_incremental["Country_Code"].map(COUNTRY_MAP)

print("\n" + "=" * 70)
print("GOOGLE TRENDS INPUT CHECK")
print("=" * 70)
print(f"Rows loaded       : {len(df_incremental):,}")
print(f"First week        : {df_incremental['Week_Start'].min().date()}")
print(f"Last week         : {df_incremental['Week_Start'].max().date()}")
print(f"Number of weeks   : {df_incremental['Week_Start'].nunique()}")
print("=" * 70)

# Remove duplicate Country / Category / Week combinations
df_incremental = (
    df_incremental
    .sort_values("Week_Start")
    .drop_duplicates(
        subset=["Week_Start", "Country_Code", "Category"],
        keep="last"
    )
)


# ------------------------------------------------------------------
# 7. Load Historical Database Baseline
# ------------------------------------------------------------------
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="!@Mstrong@1234",
    database="GDELTTrends",
)

query = """
SELECT
    Week_Start,
    Country_Code,
    Country_Name,
    Category,
    Search_Interest AS Baseline_Interest
FROM final_master_dataset;
"""

df_baseline = pd.read_sql(query, connection)
connection.close()

# Clean historical data
df_baseline["Week_Start"] = pd.to_datetime(df_baseline["Week_Start"], errors="coerce")
df_baseline["Baseline_Interest"] = pd.to_numeric(df_baseline["Baseline_Interest"], errors="coerce")

df_baseline = df_baseline.dropna(
    subset=["Week_Start", "Country_Code", "Category", "Baseline_Interest"]
)

# Fill Country_Name if missing in baseline
df_baseline["Country_Name"] = df_baseline["Country_Name"].fillna(df_baseline["Country_Code"].map(COUNTRY_MAP))

df_baseline = (
    df_baseline
    .drop_duplicates(
        subset=["Week_Start", "Country_Code", "Category"],
        keep="last"
    )
)


# ------------------------------------------------------------------
# 8. Identify Historical Boundary
# ------------------------------------------------------------------
historical_end_date = df_baseline["Week_Start"].max()

print("\n" + "=" * 70)
print("DATABASE CHECK")
print("=" * 70)
print(f"Latest historical week   : {historical_end_date.date()}")
print(f"Google Trends first week : {df_incremental['Week_Start'].min().date()}")
print(f"Google Trends last week  : {df_incremental['Week_Start'].max().date()}")
print("=" * 70)


# ------------------------------------------------------------------
# 9. Identify ALL Overlap Weeks
# ------------------------------------------------------------------
df_overlap = pd.merge(
    df_baseline[["Week_Start", "Country_Code", "Category", "Baseline_Interest"]],
    df_incremental[["Week_Start", "Country_Code", "Category", "Search_Interest"]],
    on=["Week_Start", "Country_Code", "Category"],
    how="inner"
).rename(columns={"Search_Interest": "New_Interest"})

print("\n" + "=" * 70)
print("OVERLAP CHECK")
print("=" * 70)
print(f"Total overlap rows : {len(df_overlap):,}")
print(f"Overlap weeks      : {df_overlap['Week_Start'].nunique()}")
print("=" * 70)


# ------------------------------------------------------------------
# 10. Keep Latest N Overlap Weeks
# ------------------------------------------------------------------
OVERLAP_WEEKS = 8

df_overlap = df_overlap.sort_values(["Country_Code", "Category", "Week_Start"])
df_overlap = (
    df_overlap
    .groupby(["Country_Code", "Category"], group_keys=False)
    .tail(OVERLAP_WEEKS)
)


# ------------------------------------------------------------------
# 11. Calculate Weekly Scaling Ratios
# ------------------------------------------------------------------
valid_overlap = df_overlap[
    (df_overlap["Baseline_Interest"] > 0) &
    (df_overlap["New_Interest"] > 0)
].copy()

valid_overlap["Weekly_Ratio"] = (
    valid_overlap["Baseline_Interest"] / valid_overlap["New_Interest"]
)


# ------------------------------------------------------------------
# 12. Calculate Median Ratio Per Country + Category
# ------------------------------------------------------------------
df_ratios = (
    valid_overlap
    .groupby(["Country_Code", "Category"], as_index=False)
    .agg(
        Ratio=("Weekly_Ratio", "median"),
        Overlap_Count=("Weekly_Ratio", "count")
    )
)


# ------------------------------------------------------------------
# 13. Validate Ratios
# ------------------------------------------------------------------
MIN_OVERLAP_WEEKS = 3

df_ratios_valid = df_ratios[
    df_ratios["Overlap_Count"] >= MIN_OVERLAP_WEEKS
].copy()


# ------------------------------------------------------------------
# 14. Attach Scaling Ratios
# ------------------------------------------------------------------
df_scaled = pd.merge(
    df_incremental,
    df_ratios_valid[["Country_Code", "Category", "Ratio", "Overlap_Count"]],
    on=["Country_Code", "Category"],
    how="left"
)


# ------------------------------------------------------------------
# 15. Apply Normalization
# ------------------------------------------------------------------
df_scaled["Normalized_Search_Interest"] = (
    (df_scaled["Search_Interest"] * df_scaled["Ratio"]).round(2)
)


# ------------------------------------------------------------------
# 16. Combine Historical Baseline & Normalized Incremental Data
# ------------------------------------------------------------------
df_baseline_full = df_baseline.rename(columns={"Baseline_Interest": "Search_Interest"}).copy()

df_incremental_normalized = (
    df_scaled.drop(columns=["Search_Interest", "Ratio", "Overlap_Count"])
    .rename(columns={"Normalized_Search_Interest": "Search_Interest"})
)

# Filter out un-normalized series rows
df_incremental_normalized = df_incremental_normalized[
    df_incremental_normalized["Search_Interest"].notna()
].copy()

# Combine datasets into a continuous series
df_full = pd.concat([df_baseline_full, df_incremental_normalized], ignore_index=True)

# Ensure metadata completeness across all records
df_full["Country_Name"] = df_full["Country_Name"].fillna(df_full["Country_Code"].map(COUNTRY_MAP))

# Clean duplicates at boundary, keeping normalized incremental values
df_full = df_full.sort_values("Week_Start").drop_duplicates(
    subset=["Country_Code", "Category", "Week_Start"], 
    keep="last"
)

# Ensure strict ordering prior to feature generation
df_full = df_full.sort_values(by=["Country_Code", "Category", "Week_Start"]).reset_index(drop=True)


# ------------------------------------------------------------------
# 17. Feature Engineering Across Continuous Time Series
# ------------------------------------------------------------------
grouped = df_full.groupby(["Country_Code", "Category"])

# Search_Velocity and Search Acceleartion
df_full["Search_Velocity"] = (
    df_full.groupby(["Country_Code", "Category"])["Search_Interest"].diff()
)
df_full["Search_Acceleration"] = (
    df_full.groupby(["Country_Code", "Category"])["Search_Velocity"].diff()
)

# Lag Features (1, 2, and 4 weeks)
for lag in [1, 2, 4]:
    df_full[f"Search_Interest_lag_{lag}"] = grouped["Search_Interest"].shift(lag)

# Velocity & Acceleration Lags
if "Search_Velocity" in df_full.columns:
    df_full["Search_Velocity_lag"] = grouped["Search_Velocity"].shift(1)
    df_full["Search_Velocity_Lag1"] = grouped["Search_Velocity"].shift(1)
    df_full["Search_Velocity_Lag2"] = grouped["Search_Velocity"].shift(2)

if "Search_Acceleration" in df_full.columns:
    df_full["Search_Acceleration_lag"] = grouped["Search_Acceleration"].shift(1)

# Rolling Statistics (4-week windows)
df_full["Search_Interest_roll_mean_4w"] = grouped["Search_Interest"].transform(
    lambda x: x.shift(1).rolling(4).mean()
)
df_full["Search_Interest_roll_std_4w"] = grouped["Search_Interest"].transform(
    lambda x: x.shift(1).rolling(4).std()
)



# ------------------------------------------------------------------
# 18. Filter Down to Target Horizon (New Weeks Only)
# ------------------------------------------------------------------
df_final_incremental = df_full[df_full["Week_Start"] >= historical_end_date].copy()


# ------------------------------------------------------------------
# 19. Export Clean Features Dataset
# ------------------------------------------------------------------
df_final_incremental.to_csv("Normalized_incremental_data_with_features.csv", index=False)

print("\n" + "=" * 70)
print("✅ FEATURE ENGINEERING & NORMALIZATION COMPLETE")
print(f"✅ Exported {len(df_final_incremental):,} rows without missing Country Names or boundary NaNs.")
print("📁 Saved to: Normalized_incremental_data_with_features.csv")
print("=" * 70)


========== United_Arab_Emirates (AE) ==========

========== Australia (AU) ==========

========== Brazil (BR) ==========

========== Canada (CA) ==========

========== China (CN) ==========

========== France (FR) ==========

========== India (IN) ==========

========== Kenya (KE) ==========

========== Mexico (MX) ==========

========== Nigeria (NG) ==========

========== Singapore (SG) ==========

========== South_Africa (ZA) ==========

========== United_Kingdom (GB) ==========

========== United_States (US) ==========

GOOGLE TRENDS INPUT CHECK
Rows loaded       : 504
First week        : 2026-06-28
Last week         : 2026-09-13
Number of weeks   : 12

DATABASE CHECK
Latest historical week   : 2026-08-23
Google Trends first week : 2026-06-28
Google Trends last week  : 2026-09-13

OVERLAP CHECK
Total overlap rows : 378
Overlap weeks      : 9

✅ FEATURE ENGINEERING & NORMALIZATION COMPLETE
✅ Exported 168 rows without missing Country Names or boundary NaNs.
📁 Saved to: Normalized_inc

C:\Users\Srivatsava\AppData\Local\Temp\ipykernel_7912\2998800363.py:232: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_baseline = pd.read_sql(query, connection)


### GDELT GKG data

In [12]:
# ------------------------------------------------------------------
#  Compute Extraction Date Window
# ------------------------------------------------------------------

# Include the last recorded week again
fetch_start_date = last_recorded_date

# Today
today = datetime.now().date()

# Sunday = 6 in Python's weekday() convention
# Find the most recent COMPLETED Sunday-starting week.
# current week started Sunday 2026-09-20
# therefore the latest completed week started Sunday 2026-09-13

current_week_start = today - timedelta(
    days=(today.weekday() + 1) % 7
)

# The current week is incomplete, so use the previous Sunday
fetch_end_date = current_week_start - timedelta(days=7)


# ------------------------------------------------------------------
# Check extraction range
# ------------------------------------------------------------------

if fetch_start_date > fetch_end_date:

    print(
        f"✅ Data is already up to date as of "
        f"{last_recorded_date}. "
        f"No completed weeks available to fetch."
    )

    exit()


# ------------------------------------------------------------------
# Google Trends timeframe
# ------------------------------------------------------------------

timeframe_str = (
    f"{fetch_start_date.strftime('%Y-%m-%d')} "
    f"{fetch_end_date.strftime('%Y-%m-%d')}"
)


print(f"📅 Last recorded week : {last_recorded_date}")
print(f"📅 Current week start : {current_week_start}")
print(f"📅 Last completed week: {fetch_end_date}")
print(f"🚀 Fetching range      : {timeframe_str}\n")

📅 Last recorded week : 2026-08-23
📅 Current week start : 2026-09-20
📅 Last completed week: 2026-09-13
🚀 Fetching range      : 2026-08-23 2026-09-13



In [13]:
# ------------------------------------------------------------------
# 2. Configurations & Lookups
# ------------------------------------------------------------------
COUNTRY_MAP = {
    "US": "United_States",
    "IN": "India",
    "FR": "France",
    "NG": "Nigeria",
    "GB": "United_Kingdom",
    "CA": "Canada",
    "AU": "Australia",
    "BR": "Brazil",
    "MX": "Mexico",
    "ZA": "South_Africa",
    "KE": "Kenya",
    "SG": "Singapore",
    "AE": "United_Arab_Emirates",
    "CN": "China",
}

CATEGORY_THEME_MAP = {
    "Nutrition_Diets": [
        "HEALTH_DIET",
        "EATING",
        "FOOD_SECURITY",
        "AGRICULTURE",
    ],
    "Fitness_Wearables": ["HEALTH", "FITNESS", "TECHNOLOGY", "MED_DEVICE"],
    "Fashion_Beauty": ["FASHION", "BEAUTY", "RETAIL", "CONSUMER_GOODS"],
}

### Calculating features on GDELT data
for Search_forecasting need 'Media_Volume_roll_mean_4w'
for Market segmentation need 'Mean_Media_volume','Mean_Demand_to_hype_ratio','Mean_Net_Sentiment' 

In [14]:
# ------------------------------------------------------------------
# 0. Set GCP Credentials & Client
# ------------------------------------------------------------------
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    r"C:\Users\Srivatsava\Downloads\gdelt-506803-be80fda77bde.json"
)

client = bigquery.Client(project="gdelt-506803")

# Target Date Parameters
target_start_date = last_recorded_date
target_end_date = fetch_end_date

# Parse target dates to datetime objects
if isinstance(target_start_date, str):
    start_dt = datetime.strptime(target_start_date, "%Y-%m-%d")
elif isinstance(target_start_date, date) and not isinstance(
    target_start_date, datetime
):
    start_dt = datetime.combine(target_start_date, datetime.min.time())
else:
    start_dt = target_start_date

if isinstance(target_end_date, str):
    end_dt = datetime.strptime(target_end_date, "%Y-%m-%d")
elif isinstance(target_end_date, date) and not isinstance(
    target_end_date, datetime
):
    end_dt = datetime.combine(target_end_date, datetime.min.time())
else:
    end_dt = target_end_date

# Extend query start date back by 28 days (4 weeks) to populate rolling lookbacks
lookback_start_dt = start_dt - timedelta(days=28)

start_str = lookback_start_dt.strftime("%Y-%m-%d")
end_str = end_dt.strftime("%Y-%m-%d")
target_start_str = start_dt.strftime("%Y-%m-%d")

print(
    f"🚀 Executing GDELT BigQuery extraction with 4-week lookback buffer: "
    f"{start_str} to {end_str} (Target Horizon: {target_start_str} to {end_str})"
)

# ------------------------------------------------------------------
# 1. BigQuery SQL Query
# ------------------------------------------------------------------
query = """
SELECT
  -- Sunday-aligned week start (matches Google Trends)
  DATE_TRUNC(
    PARSE_DATE('%Y%m%d', SUBSTR(CAST(DATE AS STRING), 1, 8)),
    WEEK(SUNDAY)
  ) AS Week_Start,

  -- ISO Code (for clean joining with PyTrends)
  CASE
    WHEN V2Locations LIKE '%#US#%' THEN 'US'
    WHEN V2Locations LIKE '%#IN#%' THEN 'IN'
    WHEN V2Locations LIKE '%#DE#%' THEN 'DE'
    WHEN V2Locations LIKE '%#FR#%' THEN 'FR'
    WHEN V2Locations LIKE '%#NG#%' THEN 'NG'
    WHEN V2Locations LIKE '%#GB#%' THEN 'GB'
    WHEN V2Locations LIKE '%#CA#%' THEN 'CA'
    WHEN V2Locations LIKE '%#AU#%' THEN 'AU'
    WHEN V2Locations LIKE '%#BR#%' THEN 'BR'
    WHEN V2Locations LIKE '%#MX#%' THEN 'MX'
    WHEN V2Locations LIKE '%#ZA#%' THEN 'ZA'
    WHEN V2Locations LIKE '%#KE#%' THEN 'KE'
    WHEN V2Locations LIKE '%#SG#%' THEN 'SG'
    WHEN V2Locations LIKE '%#AE#%' THEN 'AE'
    WHEN V2Locations LIKE '%#JP#%' THEN 'JP'
    WHEN V2Locations LIKE '%#CH#%' THEN 'CN'
  END AS Country_Code,

  -- Full Country Name
  CASE
    WHEN V2Locations LIKE '%#US#%' THEN 'United_States'
    WHEN V2Locations LIKE '%#IN#%' THEN 'India'
    WHEN V2Locations LIKE '%#FR#%' THEN 'France'
    WHEN V2Locations LIKE '%#NG#%' THEN 'Nigeria'
    WHEN V2Locations LIKE '%#GB#%' THEN 'United_Kingdom'
    WHEN V2Locations LIKE '%#CA#%' THEN 'Canada'
    WHEN V2Locations LIKE '%#AU#%' THEN 'Australia'
    WHEN V2Locations LIKE '%#BR#%' THEN 'Brazil'
    WHEN V2Locations LIKE '%#MX#%' THEN 'Mexico'
    WHEN V2Locations LIKE '%#ZA#%' THEN 'South_Africa'
    WHEN V2Locations LIKE '%#KE#%' THEN 'Kenya'
    WHEN V2Locations LIKE '%#SG#%' THEN 'Singapore'
    WHEN V2Locations LIKE '%#AE#%' THEN 'United_Arab_Emirates'
    WHEN V2Locations LIKE '%#CH#%' THEN 'China'
  END AS Country_Name,

  -- Consolidated 3-Category Model
  CASE
    WHEN REGEXP_CONTAINS(
      LOWER(Themes), 
      r'diet|nutrition|vegan|vegetarian|plant-based|plant based|keto|paleo|low-carb|low carb|low-fat|low fat|gluten-free|gluten free|sugar-free|sugar free|fasting|detox|clean eating|whole food|superfood|organic|supplement|protein|whey|creatine|vitamin|mineral|probiotic|functional food|weight loss|weight-loss'
    ) THEN 'Nutrition_Diets'

    WHEN REGEXP_CONTAINS(
      LOWER(Themes), 
      r'fitness|exercise|workout|training|gym|health club|yoga|pilates|aerobics|zumba|hiit|crossfit|strength training|bodybuilding|cardio|running|jogging|cycling|swimming|sports|athletics|endurance|wearable|smartwatch|fitness tracker|tracker|step counter|activity band|heart rate|heart-rate|garmin|fitbit|apple watch|smart band'
    ) THEN 'Fitness_Wearables'

    WHEN REGEXP_CONTAINS(
      LOWER(Themes), 
      r'fashion|clothing|apparel|garment|style|trend|designer|couture|runway|luxury|fast fashion|sustainable fashion|eco-fashion|streetwear|athleisure|accessories|handbag|shoe|shoes|sneaker|sneakers|jewelry|jewellery|cosmetic|cosmetics|skincare|skin care|makeup|foundation|lipstick|mascara|eyeliner|moisturizer|serum|anti-aging|haircare|shampoo|conditioner|hair treatment|fragrance|perfume|cologne|salon|spa|beauty treatment'
    ) THEN 'Fashion_Beauty'

    ELSE 'Other'
  END AS Category,

  -- Aggregations & Sentiment Metrics
  COUNT(*) AS Media_Volume,
  ROUND(AVG(SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(0)] AS FLOAT64)), 2) AS tone_net_sentiment,
  ROUND(AVG(SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(1)] AS FLOAT64)), 2) AS tone_positive_score,
  ROUND(AVG(SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(2)] AS FLOAT64)), 2) AS tone_negative_score,
  ROUND(AVG(SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(3)] AS FLOAT64)), 2) AS tone_polarity,
  ROUND(AVG(SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(4)] AS FLOAT64)), 2) AS tone_activity_density,
  ROUND(AVG(SAFE_CAST(SPLIT(V2Tone, ',')[SAFE_OFFSET(5)] AS FLOAT64)), 2) AS tone_self_group_density,
  COUNT(DISTINCT SourceCommonName) AS Source_Diversity

FROM `gdelt-bq.gdeltv2.gkg_partitioned`

WHERE
  _PARTITIONTIME >= TIMESTAMP(@start_date)
  AND _PARTITIONTIME <= TIMESTAMP(@end_date)

  -- Global Theme Recall Filter
  AND REGEXP_CONTAINS(
    LOWER(Themes),
    r'diet|nutrition|vegan|vegetarian|plant-based|keto|paleo|low-carb|gluten-free|fasting|detox|organic|supplement|protein|vitamin|fitness|workout|gym|yoga|pilates|crossfit|running|wearable|smartwatch|tracker|garmin|fitbit|fashion|clothing|apparel|designer|luxury|streetwear|athleisure|cosmetic|skincare|makeup|fragrance|spa'
  )

  -- FIPS Country Filter
  AND (
    V2Locations LIKE '%#US#%' OR V2Locations LIKE '%#IN#%'
    OR V2Locations LIKE '%#DE#%' OR V2Locations LIKE '%#FR#%'
    OR V2Locations LIKE '%#NG#%' OR V2Locations LIKE '%#GB#%'
    OR V2Locations LIKE '%#CA#%' OR V2Locations LIKE '%#AU#%'
    OR V2Locations LIKE '%#BR#%' OR V2Locations LIKE '%#MX#%'
    OR V2Locations LIKE '%#ZA#%' OR V2Locations LIKE '%#KE#%'
    OR V2Locations LIKE '%#SG#%' OR V2Locations LIKE '%#AE#%'
    OR V2Locations LIKE '%#JP#%' OR V2Locations LIKE '%#CH#%'
  )

  AND V2Tone IS NOT NULL
  AND LENGTH(V2Tone) > 5

GROUP BY
  Week_Start,
  Country_Code,
  Country_Name,
  Category

HAVING
  Week_Start IS NOT NULL
  AND Country_Code IS NOT NULL
  AND Category != 'Other'

ORDER BY
  Country_Name,
  Category,
  Week_Start;
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("start_date", "STRING", start_str),
        bigquery.ScalarQueryParameter("end_date", "STRING", end_str),
    ]
)

# Execute query and stream to Pandas
query_job = client.query(query, job_config=job_config)
df_gdelt = query_job.to_dataframe()

df_gdelt["Week_Start"] = pd.to_datetime(df_gdelt["Week_Start"])
df_gdelt = df_gdelt.sort_values(
    by=["Country_Name", "Category", "Week_Start"]
).reset_index(drop=True)

print(f"✅ Downloaded {len(df_gdelt)} rows (including lookback window).")
df_gdelt.to_csv("gdelt_data.csv",index=False)



🚀 Executing GDELT BigQuery extraction with 4-week lookback buffer: 2026-07-26 to 2026-09-13 (Target Horizon: 2026-08-23 to 2026-09-13)


c:\Users\Srivatsava\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ Downloaded 313 rows (including lookback window).


In [17]:
# ------------------------------------------------------------------
# 2. Clean & Merge Datasets (Eliminates Duplicate _x / _y Columns)
# ------------------------------------------------------------------
df_gdelt = pd.read_csv("gdelt_data.csv")
if "df_final_incremental" in locals() or "df_final_incremental" in globals():
    # Enforce datetime types
    df_final_incremental["Week_Start"] = pd.to_datetime(df_final_incremental["Week_Start"])
    df_gdelt["Week_Start"] = pd.to_datetime(df_gdelt["Week_Start"])

    # Clean df_final_incremental to keep ONLY unique, necessary trends columns
    # Dropping existing GDELT columns if they accidentally exist in df_final_incremental
    gdelt_cols_to_drop = [col for col in df_gdelt.columns if col not in ["Week_Start", "Country_Name", "Category"] and col in df_final_incremental.columns]
    df_trends_clean = df_final_incremental.drop(columns=gdelt_cols_to_drop)

    # Clean up any trailing _x or _y suffixes if present in df_trends_clean
    df_trends_clean = df_trends_clean.loc[:, ~df_trends_clean.columns.str.endswith(('_x', '_y'))]

    # Perform a clean outer/left merge based on your primary keys
    df_merged = pd.merge(
        df_trends_clean,
        df_gdelt,
        on=["Week_Start", "Country_Name", "Category"],
        how="left"
    )
else:
    df_merged = df_gdelt.copy()

# Sort data properly before running window calculations
df_merged = df_merged.sort_values(by=["Country_Name", "Category", "Week_Start"]).reset_index(drop=True)

# ------------------------------------------------------------------
# 3. Handle Missing GDELT Media Volumes & Sentiment
# ------------------------------------------------------------------
# Missing Media_Volume or Source_Diversity means 0 media articles were found for that week
media_cols_zero_fill = ["Media_Volume", "Source_Diversity"]
for col in media_cols_zero_fill:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].fillna(0)

# Missing sentiment metrics fill with ffill() and bfill()
sentiment_cols = [
    "tone_net_sentiment", "tone_positive_score", "tone_negative_score",
    "tone_polarity", "tone_activity_density", "tone_self_group_density"
]

# Sort for chronological order
df_merged = df_merged.sort_values(["Country_Code", "Category", "Week_Start"])

for col in sentiment_cols:
    if col in df_merged.columns:
        df_merged[col] = (
            df_merged.groupby(["Country_Code", "Category"])[col]
            .transform(lambda x: x.ffill().bfill())
        )


# Synchronize Country_Code column if separated
if "Country_Code_x" in df_merged.columns and "Country_Code_y" in df_merged.columns:
    df_merged["Country_Code"] = df_merged["Country_Code_x"].combine_first(df_merged["Country_Code_y"])
    df_merged = df_merged.drop(columns=["Country_Code_x", "Country_Code_y"])

# ------------------------------------------------------------------
# 4. Continuous Time-Series Feature Engineering
# ------------------------------------------------------------------
grouped = df_merged.groupby(["Country_Name", "Category"])

# 4-Week Rolling Mean Media Volume (fills early lookbacks with min_periods=1)
df_merged["Media_Volume_roll_mean_4w"] = grouped["Media_Volume"].transform(
    lambda x: x.shift(1).rolling(window=4, min_periods=1).mean()
).fillna(0)


# ------------------------------------------------------------------
# 5. Trim Buffer Data & Export Target Dataset
# ------------------------------------------------------------------
# Filter to target execution window
df_final = df_merged[
    df_merged["Week_Start"] >= pd.to_datetime(target_start_str)
].copy()

# Format Week_Start back to string YYYY-MM-DD
df_final["Week_Start"] = df_final["Week_Start"].dt.strftime("%Y-%m-%d")

print("\n" + "=" * 70)
print("✅ GDELT & TRENDS MERGE COMPLETE (NO NULL VALUES)")
print("=" * 70)
print(f"Target Row Count   : {len(df_final):,}")
print(f"Null Values Count  :\n{df_final.isnull().sum()}")
print("=" * 70)
print(df_final.head())


✅ GDELT & TRENDS MERGE COMPLETE (NO NULL VALUES)
Target Row Count   : 168
Null Values Count  :
Week_Start                       0
Country_Name                     0
Category                         0
Search_Interest                  0
Search_Velocity                  0
Search_Acceleration              0
Search_Interest_lag_1            0
Search_Interest_lag_2            0
Search_Interest_lag_4            0
Search_Velocity_lag              0
Search_Velocity_Lag1             0
Search_Velocity_Lag2             0
Search_Acceleration_lag          0
Search_Interest_roll_mean_4w     0
Search_Interest_roll_std_4w      0
Country_Code                    16
Media_Volume                     0
tone_net_sentiment              16
tone_positive_score             16
tone_negative_score             16
tone_polarity                   16
tone_activity_density           16
tone_self_group_density         16
Source_Diversity                 0
Media_Volume_roll_mean_4w        0
dtype: int64
     Week_Start 

In [18]:
df_final.head()

,Week_Start,Country_Name,Category,Search_Interest,Search_Velocity,Search_Acceleration,Search_Interest_lag_1,Search_Interest_lag_2,Search_Interest_lag_4,Search_Velocity_lag,...,Country_Code,Media_Volume,tone_net_sentiment,tone_positive_score,tone_negative_score,tone_polarity,tone_activity_density,tone_self_group_density,Source_Diversity,Media_Volume_roll_mean_4w
132,2026-08-23,United_Arab_Emirates,Fashion_Beauty,68.56,-3.36,-6.35,71.92,68.93,68.80,2.99,...,AE,170.0,2.13,4.39,2.27,6.66,20.61,0.62,134.0,0.0
133,2026-08-30,United_Arab_Emirates,Fashion_Beauty,68.30,-0.26,3.10,68.56,71.92,70.50,-3.36,...,AE,136.0,0.41,3.71,3.31,7.02,21.02,0.53,92.0,170.0
134,2026-09-06,United_Arab_Emirates,Fashion_Beauty,72.12,3.82,4.08,68.30,68.56,68.93,-0.26,...,AE,141.0,-0.80,3.46,4.26,7.72,20.25,0.43,103.0,153.0
135,2026-09-13,United_Arab_Emirates,Fashion_Beauty,68.44,-3.68,-7.50,72.12,68.30,71.92,3.82,...,AE,17.0,3.42,4.96,1.54,6.50,18.26,0.76,14.0,149.0
137,2026-08-30,United_Arab_Emirates,Fitness_Wearables,80.25,-5.26,-4.33,85.51,86.44,88.01,-0.93,...,AE,3.0,-4.30,1.50,5.80,7.30,20.75,0.00,3.0,0.0


In [19]:
# removing country code columns form df_final
df_final = df_final.drop(columns='Country_Code')

In [20]:
# Get all columns with missing values
null_cols = df_final.columns[df_final.isnull().sum() > 0]
print(null_cols)

Index(['tone_net_sentiment', 'tone_positive_score', 'tone_negative_score',
       'tone_polarity', 'tone_activity_density', 'tone_self_group_density'],
      dtype='object')


In [21]:
for col in null_cols:
    df_final[col] = df_final[col].ffill()

In [22]:
df_final.isnull().sum()

Week_Start                      0
Country_Name                    0
Category                        0
Search_Interest                 0
Search_Velocity                 0
Search_Acceleration             0
Search_Interest_lag_1           0
Search_Interest_lag_2           0
Search_Interest_lag_4           0
Search_Velocity_lag             0
Search_Velocity_Lag1            0
Search_Velocity_Lag2            0
Search_Acceleration_lag         0
Search_Interest_roll_mean_4w    0
Search_Interest_roll_std_4w     0
Media_Volume                    0
tone_net_sentiment              0
tone_positive_score             0
tone_negative_score             0
tone_polarity                   0
tone_activity_density           0
tone_self_group_density         0
Source_Diversity                0
Media_Volume_roll_mean_4w       0
dtype: int64

In [26]:
path = r'E:\\ML Project Internship\\ML-Project\\data\\processed\\'
df_final.to_csv(path + 'user_input_Merged_data.csv',index= False)

### Holidays and Macro data

In [27]:
# Target Date Parameters
target_start_date = last_recorded_date
target_end_date = fetch_end_date

In [31]:
from datetime import timedelta
import holidays
import pandas as pd

# ------------------------------------------------------------------
# 1. Country Name to ISO Alpha-2 Mapping for Python `holidays`
# ------------------------------------------------------------------
COUNTRY_MAP = {
    "United_States": "US",
    "India": "IN",
    "France": "FR",
    "Nigeria": "NG",
    "United_Kingdom": "GB",
    "Canada": "CA",
    "Australia": "AU",
    "Brazil": "BR",
    "Mexico": "MX",
    "South_Africa": "ZA",
    "Kenya": "KE",
    "Singapore": "SG",
    "United_Arab_Emirates": "AE",
    "China": "CN",
    "Germany": "DE",
    "Japan": "JP",
}

# ------------------------------------------------------------------
# 2. Holiday Feature Calculation Function
# ------------------------------------------------------------------
# Pre-cache country holiday instances to avoid re-instantiating inside loops
years_to_cover = df_final["Week_Start"].astype(str).str[:4].unique().astype(int)
holiday_dict = {}

for country_name, iso_code in COUNTRY_MAP.items():
    try:
        holiday_dict[country_name] = holidays.country_holidays(
            iso_code, years=years_to_cover
        )
    except NotImplementedError:
        # Fallback if a specific country code isn't supported in holidays library
        holiday_dict[country_name] = {}


def get_weekly_holidays(row):
    country = row["Country_Name"]
    week_start = row["Week_Start_dt"]

    # Retrieve country holiday calendar
    country_holidays = holiday_dict.get(country, {})
    if not country_holidays:
        return 0

    # Count holidays within the 7-day window [Week_Start, Week_Start + 6 days]
    week_days = [week_start + timedelta(days=i) for i in range(7)]
    count = sum(1 for day in week_days if day in country_holidays)

    return count


# ------------------------------------------------------------------
# 3. Apply Feature Engineering to DataFrame
# ------------------------------------------------------------------
# Temporarily ensure Week_Start is datetime format
df_final["Week_Start_dt"] = pd.to_datetime(df_final["Week_Start"])

# Calculate holiday_count across the 7-day week
df_final["holiday_count"] = df_final.apply(get_weekly_holidays, axis=1)

# Calculate binary indicator is_holiday
df_final["is_holiday"] = (df_final["holiday_count"] > 0).astype(int)

# Clean up helper column
df_final = df_final.drop(columns=["Week_Start_dt"])

print("\n" + "=" * 70)
print("✅ HOLIDAY FEATURES SUCCESSFULLY CREATED (NO NULL VALUES)")
print("=" * 70)
print(
    f"Holiday Feature Nulls:\n{df_final[['holiday_count', 'is_holiday']].isnull().sum()}"
)
print("=" * 70)
print(
    df_final[
        ["Week_Start", "Country_Name", "Category", "holiday_count", "is_holiday"]
    ].head(10)
)


✅ HOLIDAY FEATURES SUCCESSFULLY CREATED (NO NULL VALUES)
Holiday Feature Nulls:
holiday_count    0
is_holiday       0
dtype: int64
     Week_Start          Country_Name           Category  holiday_count  \
132  2026-08-23  United_Arab_Emirates     Fashion_Beauty              1   
133  2026-08-30  United_Arab_Emirates     Fashion_Beauty              0   
134  2026-09-06  United_Arab_Emirates     Fashion_Beauty              0   
135  2026-09-13  United_Arab_Emirates     Fashion_Beauty              0   
137  2026-08-30  United_Arab_Emirates  Fitness_Wearables              0   
138  2026-09-06  United_Arab_Emirates  Fitness_Wearables              0   
140  2026-08-23  United_Arab_Emirates    Nutrition_Diets              1   
141  2026-08-30  United_Arab_Emirates    Nutrition_Diets              0   
142  2026-09-06  United_Arab_Emirates    Nutrition_Diets              0   
143  2026-09-13  United_Arab_Emirates    Nutrition_Diets              0   

     is_holiday  
132           1  
133   

In [33]:
path = r'E:\\ML Project Internship\\ML-Project\\data\\processed\\'
df_final.to_csv(path + 'user_input_Merged_data.csv',index= False)

In [3]:
df_final = pd.read_csv("E:\\ML Project Internship\\ML-Project\\data\\processed\\user_input_Merged_data.csv")

In [4]:
import pandas as pd
import requests

# ------------------------------------------------------------------
# 1. Country ISO Mapping & World Bank Indicator Codes
# ------------------------------------------------------------------
COUNTRY_ISO_MAP = {
    "United_States": "USA",
    "India": "IND",
    "France": "FRA",
    "Nigeria": "NGA",
    "United_Kingdom": "GBR",
    "Canada": "CAN",
    "Australia": "AUS",
    "Brazil": "BRA",
    "Mexico": "MEX",
    "South_Africa": "ZAF",
    "Kenya": "KEN",
    "Singapore": "SGP",
    "United_Arab_Emirates": "ARE",
    "China": "CHN",
    "Germany": "DEU",
    "Japan": "JPN",
}

INDICATORS = {
    "Inflation_Rate": "FP.CPI.TOTL.ZG",
    "Internet_Penetration": "IT.NET.USER.ZS",
    "GDP_Per_Capita": "NY.GDP.PCAP.CD",
}

# ------------------------------------------------------------------
# 2. Fetch World Bank Indicators
# ------------------------------------------------------------------
country_codes_str = ";".join(COUNTRY_ISO_MAP.values())
wb_records = []

print("🌐 Fetching latest available World Bank indicators...")

for col_name, indicator_code in INDICATORS.items():
    # Fetch historical window (2015 to current)
    url = (
        f"http://api.worldbank.org/v2/country/{country_codes_str}/indicator/{indicator_code}"
        f"?date=2015:2026&format=json&per_page=1000"
    )

    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            data = response.json()
            if len(data) > 1 and data[1]:
                for entry in data[1]:
                    val = entry.get("value")
                    if val is not None:
                        wb_records.append(
                            {
                                "ISO3": entry["countryiso3code"],
                                "Year": int(entry["date"]),
                                "Metric": col_name,
                                "Value": float(val),
                            }
                        )
    except Exception as e:
        print(f"⚠️ Warning: Failed fetching {col_name}: {e}")

# ------------------------------------------------------------------
# 3. Extract Latest Valid Value Per Country
# ------------------------------------------------------------------
df_wb_raw = pd.DataFrame(wb_records)

# Reverse map ISO3 back to Country_Name
iso_to_country = {v: k for k, v in COUNTRY_ISO_MAP.items()}
df_wb_raw["Country_Name"] = df_wb_raw["ISO3"].map(iso_to_country)

# Sort by Year ascending so the last entry for each country is the most recent
df_wb_raw = df_wb_raw.sort_values(by="Year")

# Get the latest available value for each Country and Metric
df_latest_macro = (
    df_wb_raw.groupby(["Country_Name", "Metric"])["Value"]
    .last()
    .unstack()
    .reset_index()
)

# ------------------------------------------------------------------
# 4. Merge Latest Macro Metrics into Main DataFrame
# ------------------------------------------------------------------
macro_cols = ["Inflation_Rate", "Internet_Penetration", "GDP_Per_Capita"]

# Ensure existing columns are dropped prior to merging to prevent duplicates
df_final = df_final.drop(columns=[c for c in macro_cols if c in df_final.columns], errors="ignore")

# Merge strictly on Country_Name
df_final = pd.merge(df_final, df_latest_macro, on="Country_Name", how="left")

# Fallback: Fill any remaining unmapped country values with 0
for col in macro_cols:
    if col in df_final.columns:
        df_final[col] = df_final[col].fillna(0.0)
    else:
        df_final[col] = 0.0

print("\n" + "=" * 70)
print("✅ WORLD BANK MACROECONOMIC FEATURES SUCCESSFULLY ADDED")
print("=" * 70)
print(f"Null Values Count  :\n{df_final[macro_cols].isnull().sum()}")
print("=" * 70)
print(
    df_final[
        ["Week_Start", "Country_Name", "Inflation_Rate", "Internet_Penetration", "GDP_Per_Capita"]
    ].head(10)
)

🌐 Fetching latest available World Bank indicators...

✅ WORLD BANK MACROECONOMIC FEATURES SUCCESSFULLY ADDED
Null Values Count  :
Inflation_Rate          0
Internet_Penetration    0
GDP_Per_Capita          0
dtype: int64
   Week_Start          Country_Name  Inflation_Rate  Internet_Penetration  \
0  2026-08-23  United_Arab_Emirates        1.250865                 100.0   
1  2026-08-30  United_Arab_Emirates        1.250865                 100.0   
2  2026-09-06  United_Arab_Emirates        1.250865                 100.0   
3  2026-09-13  United_Arab_Emirates        1.250865                 100.0   
4  2026-08-30  United_Arab_Emirates        1.250865                 100.0   
5  2026-09-06  United_Arab_Emirates        1.250865                 100.0   
6  2026-08-23  United_Arab_Emirates        1.250865                 100.0   
7  2026-08-30  United_Arab_Emirates        1.250865                 100.0   
8  2026-09-06  United_Arab_Emirates        1.250865                 100.0   
9  2026-0

In [6]:
path = r'E:\\ML Project Internship\\ML-Project\\data\\processed\\'
df_final.to_csv(path + 'user_input_Merged_data.csv',index= False)

### Creating seperate data files for each model so they can be used directly by fastapi code 

In [9]:
columns = df_final.columns

In [8]:
Search_Intrest_forecasting_features = ['Week_Start', 'Country_Name', 'Category', 'Search_Interest',
       'Inflation_Rate', 'GDP_Per_Capita', 'holiday_count',
       'Search_Interest_lag_1', 'Search_Interest_lag_2',
       'Search_Interest_lag_4', 'Search_Velocity_lag',
       'Search_Acceleration_lag', 'Search_Interest_roll_mean_4w',
       'Search_Interest_roll_std_4w', 'Media_Volume_roll_mean_4w', 'sin_week',
       'cos_week']

In [10]:
set(Search_Intrest_forecasting_features)-set(columns) 

{'cos_week', 'sin_week'}

### Calculating pending features for Search_intrest forecasting

In [12]:
# Week_Start is datetime
df_final["Week_Start"] = pd.to_datetime(df_final["Week_Start"])

# Now you can safely use .dt
week_of_year = df_final["Week_Start"].dt.isocalendar().week
df_final['sin_week'] = np.sin(2 * np.pi * week_of_year / 52)
df_final['cos_week'] = np.cos(2 * np.pi * week_of_year / 52)

In [20]:
Search_Intrest_forecasting_features_df = df_final[Search_Intrest_forecasting_features].copy()

In [21]:
path = r'E:\\ML Project Internship\\ML-Project\\data\\processed\\'
Search_Intrest_forecasting_features_df.to_csv(path + "Search_Intrest_forecasting_features.csv",index=False)

### Calculating pending features for Demand_to_hype_Divergence_predcition

In [41]:
Demand_ot_hype_Divergence_score_features = ['Country_Name', 'Category','Search_Velocity', 'Search_Acceleration', 'Media_Volume',
       'tone_net_sentiment', 'tone_polarity', 'tone_activity_density',
       'Source_Diversity', 'Inflation_Rate', 'Internet_Penetration',
       'holiday_count', 'Search_Velocity_Lag1', 'Search_Velocity_Lag2']

In [18]:
set(Demand_ot_hype_Divergence_score_features) - set(columns)

set()

### No need to caluclate fetures already exist

In [42]:
path = r'E:\\ML Project Internship\\ML-Project\\data\\processed\\'
Demand_ot_hype_Divergence_score_features_df= df_final[Demand_ot_hype_Divergence_score_features].copy()
Demand_ot_hype_Divergence_score_features_df.to_csv(path + "Demand_ot_hype_Divergence_score_features.csv",index=False)

### Calculating pending features for Market_segmentation

In [24]:
Market_segmentation_features = ['Country_Name', 'Category', 'Mean_Search_Interest', 'Mean_Media_Volume',
       'Mean_Demand_to_Hype_Ratio', 'Search_Interest_Trend_Slope',
       'Mean_Net_Sentiment', 'Mean_GDP_Per_Capita']

In [25]:
mean_search = (df_final.groupby(["Country_Name", "Category"])["Search_Interest"].mean().reset_index(name="Mean_Search_Interest"))

In [26]:
# Create a new DataFrame for market segmentation
market_segmentation = mean_search.copy()

In [33]:
Mean_Media_Volume = (df_final.groupby(["Country_Name", "Category"])["Media_Volume"].mean().reset_index(name="Mean_Media_Volume"))
market_segmentation = pd.merge(market_segmentation, Mean_Media_Volume, on=["Country_Name", "Category"])

In [30]:
## Calculating Demand to Hype Ratio
df_final['Demand_to_Hype_Ratio'] = (df_final['Search_Interest'] /(df_final['Media_Volume'] + 1))

In [34]:
Mean_Demand_to_Hype_Ratio = (df_final.groupby(["Country_Name", "Category"])["Demand_to_Hype_Ratio"].mean().reset_index(name="Mean_Demand_to_Hype_Ratio"))
market_segmentation = pd.merge(market_segmentation, Mean_Demand_to_Hype_Ratio , on=["Country_Name", "Category"])

In [35]:
def calculate_slope(group):
    group = group.sort_values("Week_Start")
    x = (group["Week_Start"] - group["Week_Start"].min()).dt.days // 7 + 1
    y = group["Search_Interest"]

    x_mean, y_mean = x.mean(), y.mean()
    denominator = np.sum((x - x_mean) ** 2)

    return (
        np.sum((x - x_mean) * (y - y_mean)) / denominator
        if denominator != 0
        else 0
    )

Search_Interest_Trend_Slope = (
    df_final.groupby(["Country_Name", "Category"])
    .apply(calculate_slope)
    .reset_index(name="Search_Interest_Trend_Slope")
)
market_segmentation = pd.merge(market_segmentation, Search_Interest_Trend_Slope, on=["Country_Name", "Category"])

C:\Users\Srivatsava\AppData\Local\Temp\ipykernel_4236\3205020389.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calculate_slope)


In [37]:
mean_net_sentiment = (
    df_final.groupby(["Country_Name", "Category"])["tone_net_sentiment"]
    .mean()
    .reset_index(name="Mean_Net_Sentiment")
)
market_segmentation = pd.merge(market_segmentation, mean_net_sentiment, on=["Country_Name", "Category"])

In [38]:
mean_gdp_per_capita = (
    df_final.groupby(["Country_Name", "Category"])["GDP_Per_Capita"]
    .mean()
    .reset_index(name="Mean_GDP_Per_Capita")
)
market_segmentation = pd.merge(market_segmentation, mean_gdp_per_capita, on=["Country_Name", "Category"])

In [40]:
path = r'E:\\ML Project Internship\\ML-Project\\data\\processed\\'
market_segmentation.to_csv(path +'market_segmentation.csv',index=False)